In [23]:
import pandas as pd
import numpy as np
from sklearn.utils import shuffle


############## LOAD DATA #######################
# Load files with person labels
files_and_people = [
    ('glove_data_mqtt_3.csv', 'hafiz'),
    ('glove_data_mqtt_4.csv', 'shi_xiang'),
    ('glove_data_mqtt_5.csv', 'webster'),
    ('glove_data_mqtt_6.csv', 'cayenne'),
    ('glove_data_mqtt_7.csv', 'cayenne'),
    ('glove_data_mqtt_8.csv', 'webster'),
    ('glove_data_mqtt_9.csv', 'webster'),
    ('glove_data_mqtt_10.csv', 'hafiz'),
    ('glove_data_mqtt_11.csv', 'shi_xiang'),
    ('glove_data_mqtt_12.csv', 'cayenne'),
    ('glove_data_mqtt_13.csv', 'webster'),
    ('glove_data_mqtt_14.csv', 'hafiz'),
    ('glove_data_mqtt_15.csv', 'shi_xiang'),
    ('glove_data_mqtt_16.csv', 'webster'),
]

# Load and track person IDs
dfs = []
person_ids = []

for filename, person in files_and_people:
    df_temp = pd.read_csv(f'data/{filename}')
    dfs.append(df_temp)
    person_ids.extend([person] * len(df_temp))

# Merge all data
df = pd.concat(dfs, ignore_index=True)
df.columns = df.columns.str.strip()
person_ids = np.array(person_ids)

print(f"Total samples: {len(df)}")
print(f"Person distribution:")
for person in ['hafiz', 'shi_xiang', 'webster', 'cayenne']:
    count = np.sum(person_ids == person)
    print(f"  {person}: {count} samples")

############## PRE PROCESSING #######################
y = df["LABEL"].to_numpy()
df_features = df.drop(columns=["LABEL"])

feature_names = ["roll", "pitch", "yaw", "ax", "ay", "az", "t0", "t1", "f0", "f1", "f2", "f3"]
ignored_features = ["yaw", "t1"]
feature_names = [f for f in feature_names if f not in ignored_features]
num_timesteps = 100
num_features = len(feature_names)

columns_ordered = []
for i in range(1, num_timesteps + 1):
    for f in feature_names:
        columns_ordered.append(f"{f}_{i}")

df_features = df_features[columns_ordered]
data = df_features.to_numpy(dtype=np.float32)
X = data.reshape((-1, num_timesteps, num_features))

############## FEATURE EXTRACTION #########################
def extract_features_new(X):
    """
    Simple feature extraction: median + finger ratios + IMU deltas
    X: shape (num_samples, timesteps, num_features)
    Feature order: ["roll", "pitch", "ax", "ay", "az", "t0", "f0", "f1", "f2", "f3", "f4"]
    (yaw and t1 already removed)
    
    Returns: X_features (num_samples, num_features)
    """
    # Extract multiple statistical features
    X_median = np.median(X, axis=1)
    X_mean = np.mean(X, axis=1)
    X_std = np.std(X, axis=1)
    X_max = np.max(X, axis=1)
    X_min = np.min(X, axis=1)
    
    # Combine features
    X_features = np.hstack([X_median, X_mean, X_std, X_max, X_min])
   
    return X_features

# Feature extraction
X = np.median(X, axis=1)
# X = extract_features_new(X)

print("✅ X shape:", X.shape)
print("✅ y shape:", y.shape)
print("✅ person_ids shape:", person_ids.shape)

Total samples: 906
Person distribution:
  hafiz: 169 samples
  shi_xiang: 94 samples
  webster: 550 samples
  cayenne: 93 samples
✅ X shape: (906, 10)
✅ y shape: (906,)
✅ person_ids shape: (906,)


In [28]:
from sklearn.model_selection import cross_val_score, LeaveOneGroupOut, GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC

# === COMPARISON: Regular CV vs Person-Aware CV ===

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', LinearSVC(C=0.100, class_weight='balanced', random_state=50, max_iter=5000))
])

print("\n" + "="*70)
print("COMPARISON: Different Cross-Validation Strategies")
print("="*70)

# 1. Repeated K fold stratified
from sklearn.model_selection import RepeatedStratifiedKFold
cv_leaky = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=13)
scores_leaky = cross_val_score(pipeline, X, y, cv=cv_leaky, scoring='accuracy')

print("\nRepeatedStratifiedKFold:")
print(f"   Mean: {scores_leaky.mean():.4f} ± {scores_leaky.std():.4f}")
print(f"   Range: [{scores_leaky.min():.4f}, {scores_leaky.max():.4f}]")
print("Training and test sets share the same people")

# 2. Leave-One-Person-Out (realistic)
cv_lopo = LeaveOneGroupOut()
scores_lopo = cross_val_score(pipeline, X, y, cv=cv_lopo, groups=person_ids, scoring='accuracy')

print("\nLEAVE-ONE-PERSON-OUT (LOPO):")
print(f"   Mean: {scores_lopo.mean():.4f} ± {scores_lopo.std():.4f}")
print(f"   Range: [{scores_lopo.min():.4f}, {scores_lopo.max():.4f}]")
print("   Per-person test accuracy:")
for i, person in enumerate(['hafiz', 'shi_xiang', 'webster', 'cayenne']):
    print(f"     {person}: {scores_lopo[i]:.4f}")

# 3. Group K-Fold (alternative - still person-aware)
cv_group = GroupKFold(n_splits=4)  # 4 people, so 4 folds max
scores_group = cross_val_score(pipeline, X, y, cv=cv_group, groups=person_ids, scoring='accuracy')

print("\n GROUP K-FOLD:")
print(f"   Mean: {scores_group.mean():.4f} ± {scores_group.std():.4f}")
print(f"   Range: [{scores_group.min():.4f}, {scores_group.max():.4f}]")

print("\n" + "="*70)
print("DIAGNOSIS:")
print("="*70)
gap = scores_leaky.mean() - scores_lopo.mean()
print(f"Overfitting gap: {gap*100:.1f}%")
if gap > 0.15:
    print("SEVERE person-dependent overfitting detected!")
elif gap > 0.08:
    print("MODERATE person-dependent overfitting")
else:
    print("Model generalizes reasonably across people")


COMPARISON: Different Cross-Validation Strategies

RepeatedStratifiedKFold:
   Mean: 0.9565 ± 0.0157
   Range: [0.9171, 0.9779]
Training and test sets share the same people

LEAVE-ONE-PERSON-OUT (LOPO):
   Mean: 0.8569 ± 0.0357
   Range: [0.8065, 0.9043]
   Per-person test accuracy:
     hafiz: 0.8065
     shi_xiang: 0.8462
     webster: 0.9043
     cayenne: 0.8709

 GROUP K-FOLD:
   Mean: 0.8569 ± 0.0357
   Range: [0.8065, 0.9043]

DIAGNOSIS:
Overfitting gap: 10.0%
MODERATE person-dependent overfitting


In [32]:
import os
import matplotlib.pyplot as plt

# Fit final model
final_model = Pipeline([
    ('scaler', StandardScaler()),
    ('svc', LinearSVC(C=0.100, class_weight='balanced', random_state=50, max_iter=5000))
])
final_model.fit(X, y)

# After training your model
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC

scaler = final_model.named_steps['scaler']
svc = final_model.named_steps['svc']

mean = scaler.mean_
std = scaler.scale_
coef = svc.coef_
intercept = svc.intercept_

# Create directory
os.makedirs('test_data_svc', exist_ok=True)

# Save model parameters (NO classes array)
np.savez('test_data_svc/svc_model_mqtt_median_906_50features_Cpt1.npz',
         coef=svc.coef_,
         intercept=svc.intercept_,
         mean=scaler.mean_,
         std=scaler.scale_)

# Save class labels as a simple text file
with open('test_data_svc/svc_classes.txt', 'w') as f:
    for label in svc.classes_:
        f.write(f"{label}\n")

print("Model saved")
print(f"  Parameters: svc_model.npz")
print(f"  Classes: svc_classes.txt")
print(f"  Classes are: {list(svc.classes_)}")

Model saved
  Parameters: svc_model.npz
  Classes: svc_classes.txt
  Classes are: ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'REST']
